Pripremili smo odabrani data set: **Data Science Salaries 2023** za analizu, prikazali smo informacije kojim raspolažemo i njihove tipove podataka

In [ ]:
!git clone https://github.com/samiracatic/VIS24/
!pip install country_converter

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statistics
import plotly.express as px
import country_converter as coco
import plotly.figure_factory as ff
import plotly.graph_objects as go
import warnings
import nltk
from scipy.stats import shapiro
from scipy.stats import kstest
from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency
from statsmodels.formula.api import ols
from itertools import combinations
from scipy import stats
from scipy.stats import norm
from mpl_toolkits import mplot3d
from prettytable import PrettyTable
from google.colab import drive
from wordcloud import WordCloud
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from plotnine import *
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/VIS24/salaries.csv')

In [ ]:
df.info()

In [ ]:
df.head(10)

Pretvaranje vrijednosti u pojedinačne varijable radi olakšane manipulacije kao i prikaz različitih vrijednosti kategoričkih varijabli.

In [ ]:
work_year=df['work_year']
experience=df['experience_level']
employment_type=df['employment_type']
job=df['job_title']
salary_u=df['salary']
salary=df['salary_in_usd']
currency=df['salary_currency']
residence=df['employee_residence']
remote_ratio=df['remote_ratio']
location=df['company_location']
company_size=df['company_size']

In [ ]:
print("Moguce kategorijske vrijednosti za nivo iskustva: ")
jedinstveni_niz = list(set(experience))
for vrijednost in jedinstveni_niz:
    print(vrijednost)
print("\n")
print("Moguce kategorijske vrijednosti za tip zaposljenja: ")
jedinstveni_niz = list(set(employment_type))
for vrijednost in jedinstveni_niz:
    print(vrijednost)
print("\n")
print("Moguce kategorijske vrijednosti za vrstu posla koju obavlja: ")
jedinstveni_niz = list(set(job))
for vrijednost in jedinstveni_niz:
    print(vrijednost)

Moguće kategorijske vrijednosti za nivo iskustva:
* "MI": Srednji nivo/Intermediarni ("Mid-level/Intermediate")
* "SE": Stariji izvršni nivo ("Senior Executive")
* "EX": Izvršni nivo ("Executive level")
* "EN": Nivo početnika ("Entry-level")

Moguće kategorijske vrijednosti za tip zaposljenja:
* "FL": Puno radno vreme ("Full-time")
* "FT": Slobodno zanimanje/Ugovor ("Freelance/Contract")
* "PT": Nepuno radno vreme ("Part-time")
* "CT": Konsultant ("Consultant")

#**DESKRIPTIVNA STATISTIKA**

**Pitanje br. 1.**

*Kako se prosječna plata mijenja u zavisnosti od tipa zaposlenja, nivoa iskustva te radne godine, prikazana grupisanim bar grafikonom?*

In [ ]:
import plotly.express as px

prosječna_plata_po_iskustvu = df.groupby(['work_year', 'experience_level', 'employment_type'])['salary'].mean().reset_index()
fig = px.bar(prosječna_plata_po_iskustvu,
             x='work_year',
             y='salary',
             color='experience_level',
             barmode='group',
             facet_col='employment_type',
             title='Prosječna plata u odnosu na tip zaposlenja, nivo iskustva i radne godine')
fig.update_layout(yaxis_title='Prosječna plata', xaxis_title='Radna godina')
fig.show()
print('\n')



Analiza:
* Tip zaposlenja: Grafikon prikazuje prosječne plate u odnosu na godinu rada, nivo iskustva i tip zaposlenja (CT, FT, PT, FL).
* Nivo iskustva: Boje na grafikonu predstavljaju različite nivoe iskustva:
1. EN (Entry-level)
2. EX (Executive)
3. MI (Mid-level)
4. SE (Senior-level)
* Radna godina: Podaci su grupisani po godinama rada za
svaki tip zaposlenja (2020, 2021, 2022, 2023).
Zaključci:
* Executive pozicije: Zaposlenici na izvršnim pozicijama (EX) imaju najviše
prosječne plate u većini tipova zaposlenja i godine rada.
* Rast plata s iskustvom: Uopšteno, plate rastu s nivoom iskustva, pri čemu
seniori (SE) i mid-level (MI) zaposlenici zarađuju više od početnika (EN).
* Različiti tipovi zaposlenja:
1) CT (Contract): Izvršni (EX) i mid-level (MI) zaposlenici imaju visoke
plate, posebno u 2021. godini.
2) FT (Full-time): Senior (SE) i mid-level (MI) zaposlenici imaju najviše
plate, dok entry-level (EN) i executive (EX) pozicije pokazuju manju
varijabilnost.
3) PT (Part-time): Podaci pokazuju manje plate u poređenju s full-time i
contract zaposlenicima, s većim varijacijama između nivoa iskustva.
4) FL (Freelance): Prosječne plate za freelance pozicije pokazuju velike
varijacije, s visokim platama za mid-level (MI) pozicije.


Sproveli smo statističko računanje prosječne plate tj. ispitali smo kakva je deskriptivna statistika za platu

In [ ]:
salary_mean=statistics.mean(salary)
salary_mode=statistics.mode(salary)
salary_mediana=statistics.median(salary)
salary_varijansa=statistics.variance(salary)

print("Prosječna plata: ",salary_mean)
print("Moda prosječne plate: ",salary_mode)
print("Medijana prosječne plate: ",salary_mediana)
print("Varijansa prosječne plate: ",salary_varijansa)

**Pitanje br. 2.**

*Kako se prosječna plata razlikuje među različitim nivoima iskustva, tipovima zaposlenja i lokacijama kompanije, uzimajući u obzir godine radnog iskustva i pol zaposlenog?*

*Prije donošenja odgovora na postavljeno pitanje, sprovodimo analizu kako bi istakli razlike u različitim faktorima između muškaraca i žena. Jedan od koraka u ovoj analizi je procjena procentualnog udjela muškaraca i žena u našem skupu podataka, što omogućava procjenu relevantnosti ovih kategorija za daljnje istraživanje. Ukoliko primjetimo značajnu procentualnu razliku između ove dvije grupe, može biti neophodno proširiti analizu kako bismo osigurali adekvatan uzorak i validnost rezultata. Ovakav pristup osigurava da naša istraživanja budu pouzdana i relevantna.*

In [ ]:
import pandas as pd
total_count = len(df)
male_count = df[df['Sex'] == 'Male'].shape[0]
female_count = df[df['Sex'] == 'Female'].shape[0]
male_percentage = (male_count / total_count) * 100
female_percentage = (female_count / total_count) * 100
print(f"Procenat muškaraca: {male_percentage:.2f}%")
print(f"Procenat žena: {female_percentage:.2f}%")


*Na osnovu dobijenih podataka, možemo zaključiti da u našem skupu podataka postoji relativno ravnotežan broj muškaraca i žena, pri čemu muškarci čine oko 48.87% i žene oko 51.13% ukupnog broja zaposlenika. S obzirom na približnu ravnotežu između ove dve grupe, porediti ih može biti relevantno i korisno za analizu razlika i sličnosti u različitim faktorima kao što su performanse na poslu, zadovoljstvo poslom, ili druge relevantne varijable*

In [ ]:
import pandas as pd
average_salary = df.groupby(['experience_level', 'employment_type', 'company_location', 'Sex'])['salary'].mean().reset_index()
print(average_salary)

*Analiza:*

*Na osnovu analize podataka o platama prema različitim nivoima iskustva, tipovima zaposlenja, lokacijama kompanija i spolu zaposlenih, možemo izvući sljedeće zaključke:*

1. *Nivo Iskustva:*
   - Prosječna plata varira značajno između različitih nivoa iskustva. Seniori (SE) imaju najviše prosječne plate u većini slučajeva, dok juniori (EN) imaju najniže.

2. *Tip Zaposlenja:*
   - Zaposleni na puno radno vrijeme (FT) imaju, u prosjeku, više plate u odnosu na one koji su zaposleni na ugovor (CT) ili honorarno (FL).

3. *Lokacija Kompanije:*
   - Lokacija kompanije igra veliku ulogu u prosječnim platama. Na primjer, zaposleni u Sjedinjenim Američkim Državama (US) generalno imaju više prosječne plate u poređenju sa drugim zemljama.

4. *Spol Zaposlenog:*
   - Postoje razlike u prosječnim platama između muškaraca i žena, ali ove razlike variraju u zavisnosti od nivoa iskustva, tipa zaposlenja i lokacije kompanije. Na primjer, u nekim slučajevima žene zarađuju više od muškaraca, dok je u drugim slučajevima suprotno.

*Ovi zaključci nam pomažu da shvatimo kako različiti faktori utiču na prosječne plate i omogućavaju nam da identifikujemo područja koja zahtjevaju dodatnu analizu i potencijalne intervencije za unapređenje pravičnosti u platama.*

**Pitanje br. 3.**

*Koja zanimanja imaju najvišu platu?*

In [ ]:
salary_designation = (
    df.groupby(['salary_in_usd', 'job_title'])
    .size()
    .to_frame(name='count')
    .reset_index()
    .sort_values(by='salary_in_usd', ascending=False)
    .head(25)
)
salary_designation['salary_percentage'] = (
    salary_designation['salary_in_usd'] / salary_designation['salary_in_usd'].sum()
) * 100
fig = px.bar(
    x=salary_designation['job_title'],
    y=salary_designation['salary_in_usd'],
    text=salary_designation['salary_percentage'].apply('{:.1f}%'.format),
    color=salary_designation['salary_in_usd'],
    title='Top 25 najvećih plata'
)
fig.update_layout(
    xaxis_title="Zanimanje",
    yaxis_title="Plate",
    xaxis_tickangle=-45,
)
fig.show()

*Analiza:*

*Raznolikost pozicija: Prikazane su različite pozicije u oblasti podataka i mašinskog učenja, uključujući inženjere, naučnike, analitičare i menadžere.*

*Zaključci:*

*Najplaćenije pozicije:*

*Pozicija sa najvišom platom je "Data Analyst" sa procentualnim učešćem od 4.6% i platom od 1,239,667 USD, što ukazuje na visoku vrijednost ove pozicije u industriji.*

*Razlike u platama:*

*Postoji značajna varijabilnost u platama među top 25 pozicija. Na primjer, "AI Scientist" zarađuje 423,834 USD, što čini 4.5% ukupnih plata, dok "Data Engineer" zarađuje 324,000 USD, sa procentualnim učešćem od 3.5%.
Visoke plate za naučne pozicije:*

*Mnoge pozicije povezane s naukom o podacima i mašinskim učenjem imaju visoke plate. Na primjer:*

"Director of Data Science" - 4.0% ukupnih plata,
"Research Scientist" - 4.8% ukupnih plata,
"Applied Machine Learning Scientist" - 4.4% ukupnih plata,

*Menadžerske pozicije:*

*Menadžerske pozicije kao što su "Head of Data" i "Director of Data Science" takođe imaju visoke plate, što naglašava važnost upravljačkih uloga u ovim oblastima:*

"Head of Data" - 3.7% ukupnih plata,
"Director of Data Science" - 4.0% ukupnih plata,
Ovaj grafikon pruža vrijedne uvide u to koje su pozicije najplaćenije u industriji podataka i mašinskog učenja, ističući važnost specijalizacije i menadžerskih vještina za postizanje viših plata.

**Pitanje br. 4.**

*Kako su radnici distribuirani po državama?*

In [ ]:
country_converter = coco.CountryConverter()
iso3_codes = country_converter.convert(names=df['employee_residence'], to='ISO3')
df['employee_residence'] = iso3_codes
residence_counts = df['employee_residence'].value_counts().reset_index()
residence_counts.columns = ['ISO3', 'Count']
fig = px.choropleth(data_frame=residence_counts,
                    locations='ISO3',
                    color='Count',
                    color_continuous_scale='Turbo',
                    title='Distribucija zaposlenika po zemljama',
                    labels={'Count': 'Broj zaposlenika', 'ISO3': 'Kod zemlje'})

fig.show()

 Analiza:
1. **Geografska distribucija:**
   - Graf prikazuje globalnu distribuciju zaposlenika, s koncentracijom zaposlenika u određenim regijama.
   - Gradijent boja označava broj zaposlenika, pri čemu svjetlije boje predstavljaju veći broj zaposlenika.

2. **Detalji:**
   - Sjedinjene Američke Države imaju najvišu koncentraciju zaposlenika, označenu crvenom bojom, što ukazuje na značajan broj zaposlenika.
   - Ostale zemlje s većim brojem zaposlenika uključuju Kanadu, razne evropske zemlje (npr. Njemačka, Velika Britanija, Španija), Indiju i neke zemlje u istočnoj Aziji.
   - Neke regije, poput Afrike i dijelova Južne Amerike, imaju nižu koncentraciju zaposlenika, što je označeno tamnijim bojama na mapi.

Zaključak:
- Graf otkriva da su Sjedinjene Američke Države glavno središte zapošljavanja, s velikim brojem zaposlenika. To bi se moglo pripisati velikoj prisutnosti tehnološke industrije i brojnim radnim prilikama u zemlji.
- Ostale zemlje sa značajnim brojem zaposlenika su Kanada, Njemačka, Velika Britanija, Španija i Indija, što odražava rastuće tehnološke industrije i tržišta rada u tim zemljama.
- Niža koncentracija zaposlenika u određenim regijama može ukazivati na manje radnih prilika ili manju zastupljenost u bazi podataka.



**Pitanje br. 5.**

*Koji je prosječan prihod za svaki tip zaposlenja?*

In [ ]:
avg_salary_by_employment_type = df.groupby('employment_type')['salary_in_usd'].mean().reset_index()

fig = px.bar(avg_salary_by_employment_type, x='employment_type', y='salary_in_usd', color='employment_type', title='Prosječna plata po tipu zaposlenja')
fig.update_layout(xaxis_title="Tip zaposlenja", yaxis_title="Prosječna plata (USD)")
fig.show()

Analiza:
1. **Grafički prikaz prosječne plate po tipu zaposlenja:**
   - Graf prikazuje prosječne plate u američkim dolarima (USD) za različite tipove zaposlenja (CT, FL, FT, PT).

2. **Detalji:**
   - **FT**: Puno radno vrijeme ima najvišu prosječnu platu, koja premašuje 140,000 USD.
   - **CT**: Konsultanti zaposlenje ima drugu najvišu prosječnu platu, oko 110,000 USD.
   - **FL**: Freelanceri imaju značajno nižu prosječnu platu, koja iznosi oko 50,000 USD.
   - **PT**: Nepuno radno vrijeme ima najnižu prosječnu platu, koja je ispod 40,000 USD.

Zaključak:
- Zaposlenici s punim radnim vremenom (FT) imaju najvišu prosječnu platu, što može ukazivati na stabilnost i pogodnosti koje dolaze s punim radnim vremenom.
- Ugovorni radnici (CT) također imaju visoke plate, što može biti rezultat specijalizovanih vještina ili privremenih potreba poslodavaca.
- Slobodni radnici (FL) i zaposlenici sa skraćenim radnim vremenom (PT) imaju niže prosječne plate, što može odražavati nestalnu prirodu njihovih angažmana ili manje radne sate.


**Pitanje br. 6.**

*Kako lokacija radnika utiče na platu?*

In [ ]:
avg_salary_by_location = df.groupby('employee_residence')['salary_in_usd'].mean().reset_index()

fig = px.bar(avg_salary_by_location, x='employee_residence', y='salary_in_usd', color='employee_residence', title='Prosječna plata po lokaciji zaposlenika')
fig.update_layout(xaxis_title="Lokacija zaposlenika", yaxis_title="Prosječna plata (USD)")
fig.show()


Analiza:
1. **Grafički prikaz prosječne plate po lokaciji zaposlenika:**
   - Graf prikazuje prosječne plate u američkim dolarima (USD) za različite zemlje prebivališta zaposlenika.
   
3. **Detalji:**
   - **ISR (Izrael)** ima najvišu prosječnu platu, koja premašuje 400,000 USD.
   - **MYS (Malezija)** i **PRI (Portoriko)** također pokazuju visoke prosječne plate.
   - **US (Sjedinjene Američke Države)** i **CA (Kanada)** imaju visoke prosječne plate, što je vidljivo i na grafu.
   - Ostale zemlje s visokim prosječnim platama uključuju Kinu, Novi Zeland, Bosnu i Hercegovinu, Irsku i Dominikansku Republiku.

Zaključak:
- Postoje značajne razlike u prosječnim platama među različitim zemljama, a visoke prosječne plate u nekim zemljama mogu biti rezultat specifičnih industrija ili ekonomskih uslova.

**Pitanje br. 7.**

*Koje su države sa najviše zaposlenika sa 100% remote radom?*

In [ ]:
remote_work_by_country = df[df['remote_ratio'] == 100].groupby('employee_residence').size().reset_index(name='count')

fig = px.bar(remote_work_by_country, x='employee_residence', y='count', color='employee_residence', title='Broj zaposlenika sa 100% remote radom po državama')
fig.update_layout(xaxis_title="Država", yaxis_title="Broj zaposlenika")
fig.show()


*Radi poboljšanja vizuelne jasnoće podataka, izuzeli smo Sjedinjene Američke Države (USA) iz analize, budući da ta država značajno odudara od ostalih. Tako prilagođen grafikon omogućava preciznije sagledavanje distribucije zaposlenika po drugim zemljama, olakšavajući donošenje zaključaka i interpretaciju podataka.*

In [ ]:
import plotly.express as px
import pandas as pd
filtered_df = df[df['employee_residence'] != 'USA']
remote_work_by_country = filtered_df[filtered_df['remote_ratio'] == 100].groupby('employee_residence').size().reset_index(name='count')
fig = px.bar(remote_work_by_country, x='employee_residence', y='count', color='employee_residence', title='Broj zaposlenika sa 100% remote radom po državama')
fig.update_layout(xaxis_title="Država", yaxis_title="Broj zaposlenika")
fig.show()


Analiza:
1. **Grafički prikaz broja zaposlenika sa 100% remote radom po državama:**
   - Graf prikazuje broj zaposlenika koji rade 100% na daljinu za različite zemlje.
   
2. **Detalji:**
   - **GB (Velika Britanija)** i **CA (Kanada)** također imaju značajan broj zaposlenika koji rade na daljinu.
   - Ostale zemlje poput Indije, Španije, Njemačke, Brazila, Francuske, Grčke i Meksika također imaju zaposlenike koji rade 100% na daljinu, ali u manjem broju u usporedbi s SAD-om, GB i CA.

Zaključak:
- Sjedinjene Američke Države imaju daleko najviši broj zaposlenika koji rade 100% na daljinu.
- Velika Britanija i Kanada također imaju značajan broj zaposlenika koji rade na daljinu.



**Pitanje br. 8**

*Kako se mijenjaju plate zaposlenika u skladu s veličinom kompanije?*

In [ ]:
import pandas as pd
import plotly.figure_factory as ff
import plotly.graph_objects as go


company_size_salary = df[['company_size', 'salary_in_usd']]

small = company_size_salary[company_size_salary['company_size'] == 'S']
mid = company_size_salary[company_size_salary['company_size'] == 'M']
large = company_size_salary[company_size_salary['company_size'] == 'L']
hist_data = [small['salary_in_usd'], mid['salary_in_usd'], large['salary_in_usd']]
group_labels = ['Small', 'Mid', 'Large']
means = [small['salary_in_usd'].mean(), mid['salary_in_usd'].mean(), large['salary_in_usd'].mean()]
fig1 = ff.create_distplot(hist_data, group_labels, show_hist=False)
fig2 = go.Figure(data=px.bar(x=group_labels, y=means, color=group_labels,
                             title='Prosječna plata po veličini kompanije'))
fig1.update_layout(title='Raspodjela plata prema veličini kompanije')
fig2.update_layout(xaxis_title="Veličina kompanije", yaxis_title="Prosječna plata (k)")
fig1.show()
fig2.show()


Analiza:
1. **Grafički prikaz raspodjele plata prema veličini kompanije:**
   - Gornji graf prikazuje distribuciju plata zaposlenika u zavisnosti od veličine kompanije. Veličine kompanija su označene kao Small (male), Mid (srednje) i Large (velike).

2. **Prosječna plata po veličini kompanije:**
   - Donji graf prikazuje prosječne plate za različite veličine kompanija:
     - **Small (male kompanije)**: Prosječna plata je oko 78,227 USD.
     - **Mid (srednje kompanije)**: Prosječna plata je oko 143,131 USD.
     - **Large (velike kompanije)**: Prosječna plata je oko 118,301 USD.

3. **Detalji:**
   - **Srednje kompanije (Mid)** imaju najvišu prosječnu platu, koja premašuje 140,000 USD. Ovo je prikazano na donjem grafu gdje je srednja kompanija označena narančastom bojom.
   - **Velike kompanije (Large)** imaju drugu najvišu prosječnu platu, približno 118,301 USD, što je prikazano zelenom bojom na grafu.
   - **Male kompanije (Small)** imaju najnižu prosječnu platu, oko 78,227 USD, što je prikazano plavom bojom na grafu.

Zaključak:
- Srednje kompanije nude najviše prosječne plate svojim zaposlenicima u poređenju s malim i velikim kompanijama.
- Velike kompanije također nude konkurentne plate, ali one su niže od plata u srednjim kompanijama.
- Male kompanije imaju najniže prosječne plate, što može biti rezultat ograničenih resursa i manjih budžeta za plate u poređenju s većim kompanijama.

**Pitanje br. 9.**

*Koja je vjerovatnoća da osoba koja radi kao Data Scientist ima platu veću od 100,000 USD?*

In [ ]:

data_scientist_salaries = df[df['job_title'] == 'Data Scientist']

probability_above_100k = (data_scientist_salaries['salary_in_usd'] > 100000).mean()

print("Vjerovatnoća da osoba koja radi kao Data Scientist ima platu veću od 100,000 USD je ", probability_above_100k)


*Ova statistika ukazuje na atraktivnost i profitabilnost ove profesije, što može biti od ključnog značaja za pojedince koji razmišljaju o karijeri u ovom polju. Takođe, visok procenat onih koji prelaze ovaj prag plate može biti indikativan za tržišne trendove i rastuću potražnju za vještinama i znanjem koje posjeduju Data Scientisti.*

**Pitanje br. 10**

*Koja je razlika u platama između muškaraca i žena za iste poslova?*

In [ ]:

male_salaries = df[df['Sex'] == 'Male']
female_salaries = df[df['Sex'] == 'Female']
merged_salaries = pd.merge(male_salaries, female_salaries, on='job_title', suffixes=('_male', '_female'))
merged_salaries['salary_difference'] = merged_salaries['salary_in_usd_male'] - merged_salaries['salary_in_usd_female']
avg_salary_difference_by_job = merged_salaries.groupby('job_title')['salary_difference'].mean().reset_index()
print(avg_salary_difference_by_job)


*Dalje smo izvršili detaljniju analizu razlika u platama između muškaraca i žena za iste poslove u različitim zemljama. S obzirom na to da status žena nije ujednačen u svim zemljama, sledi vizuelni prikaz rezultata.*

In [ ]:
import pandas as pd
import plotly.express as px
male_salaries = df[df['Sex'] == 'Male']
female_salaries = df[df['Sex'] == 'Female']
avg_male_salaries_by_country = male_salaries.groupby('employee_residence')['salary_in_usd'].mean().reset_index()
avg_female_salaries_by_country = female_salaries.groupby('employee_residence')['salary_in_usd'].mean().reset_index()
fig_male = px.bar(avg_male_salaries_by_country,
                  x='employee_residence',
                  y='salary_in_usd',
                  title='Prosečne plate muškaraca po zemljama',
                  labels={'salary_in_usd': 'Prosečna plata (USD)', 'employee_residence': 'Zemlja'})
fig_female = px.bar(avg_female_salaries_by_country,
                    x='employee_residence',
                    y='salary_in_usd',
                    title='Prosečne plate žena po zemljama',
                    labels={'salary_in_usd': 'Prosečna plata (USD)', 'employee_residence': 'Zemlja'})

fig_male.show()
fig_female.show()


Analiza:
1. **Grafički prikaz prosječnih plata muškaraca po zemljama:**
   - Graf prikazuje prosječne plate muškaraca u različitim zemljama.
   - Najviše prosječne plate za muškarce prikazane su za Izrael, Portoriko i Maleziju.
2. **Grafički prikaz prosječnih plata žena po zemljama:**
   - Graf prikazuje prosječne plate žena u različitim zemljama.
   - Najviše prosječne plate za žene prikazane su za Portoriko, Sjedinjene Američke Države i Kanadu.


Zaključak:
- Postoji značajna razlika u prosječnim platama između muškaraca i žena u različitim zemljama, što može biti rezultat različitih faktora kao što su industrija, politika zapošljavanja i društveni stavovi. Prepoznavanje zemalja sa manjim platnim razlikama može pomoći u identifikaciji najboljih praksi koje se mogu implementirati na globalnom nivou kako bi se smanjila nejednakost u platama. Promovisanje jednakih mogućnosti za obrazovanje i profesionalni razvoj, kao i sprovođenje politika koje podržavaju ravnopravnost polova na radnom mjestu, ključni su koraci ka smanjenju ovih razlika.

*Daljnje istraživanje i inicijative trebaju biti usmjerene na razumijevanje i uklanjanje prepreka koje doprinose rodnim platnim razlikama, čime bi se poboljšala ekonomska ravnopravnost i omogućio pravičan pristup resursima i mogućnostima za sve.*

**Pitanje br. 11.**

*Koja je distribucija plata za top 20 poslova u SAD*

In [ ]:
import pandas as pd
import plotly.express as px

selected_country = 'US'
salaries_by_job_title = df[df['company_location'] == selected_country]
average_salaries = salaries_by_job_title.groupby('job_title')['salary_in_usd'].mean().reset_index()
top_20_job_titles = average_salaries.sort_values(by='salary_in_usd', ascending=False).head(20)['job_title']
top_20_salaries = salaries_by_job_title[salaries_by_job_title['job_title'].isin(top_20_job_titles)]
min_std_dev = 500
filtered_salaries = top_20_salaries.groupby('job_title').filter(lambda x: x['salary_in_usd'].std() >= min_std_dev)
fig = px.violin(filtered_salaries, y="salary_in_usd", x="job_title", box=True,
                title=f'Distribucija plata za različite poslove u državi {selected_country}',
                labels={'salary_in_usd': 'Plata (USD)', 'job_title': 'Naslov posla'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()



Analiza:
1. **Distribucija plata za različite poslove u državi US:**
   - Graf prikazuje distribuciju plata za različite poslove u Sjedinjenim Američkim Državama.
   - Različiti naslovi poslova su prikazani na x-osi, dok y-osa prikazuje plate u USD.

2. **Detalji:**
   - **Head of Data Science** ima najvišu prosječnu platu, koja iznosi 300,000 USD.
   - **Applied Scientist** i **Head of Data** također imaju visoke prosječne plate.
   - **Data Science Manager**, **Director of Data Science**, i **Machine Learning Infrastructure Engineer** imaju značajne plate, ali niže od pozicija poput **Head of Data Science**.
   - Postoje razlike u distribuciji plata unutar istih naslova poslova, što može biti rezultat različitih nivoa iskustva, veličine kompanija i drugih faktora.

Zaključak:
- Najviše plate u Sjedinjenim Američkim Državama su povezane s pozicijama koje uključuju vođenje i upravljanje podacima, kao što su **Head of Data Science** i **Head of Data**.
- Tehničke pozicije kao što su **Applied Scientist** i **Machine Learning Infrastructure Engineer** također imaju visoke plate, ali su nešto niže u poređenju s upravljačkim pozicijama.
- Postoje značajne razlike u platama unutar istih naslova poslova, što može ukazivati na uticaj iskustva, veličine kompanije i specifičnih odgovornosti.

**Pitanje br. 12.**

*Koji tipovi zaposlenja imaju najveći raspon plata za određeni nivo iskustva? Postoje li izuzetno visoke ili niske plate u odnosu na iskustvo i tip zaposlenja?*

In [ ]:
plt.figure(figsize=(14, 8))
sns.boxplot(x='employment_type', y='salary_in_usd', hue='experience_level', data=df)
plt.title('Raspon plata po tipovima zaposlenja i nivou iskustva')
plt.xlabel('Tip zaposlenja')
plt.ylabel('Plata (USD)')
plt.xticks(rotation=45)
plt.legend(title='Nivo iskustva')
plt.tight_layout()

plt.show()


Analiza:
1. **Raspon plata po tipovima zaposlenja i nivou iskustva:**
   - Graf prikazuje raspon plata (USD) za različite tipove zaposlenja (FT - puno radno vrijeme, CT - ugovor, FL - slobodni radnik, PT - skraćeno radno vrijeme) u odnosu na nivo iskustva (EN - početnik, MI - srednji nivo, SE - stariji nivo, EX - ekspert).

2. **Detalji:**
   - **Eksperti (EX)** u ugovornom zaposlenju (CT) imaju najvišu prosječnu platu, koja iznosi 416,000 USD.
   - **Stariji nivo (SE)** u punom radnom vremenu (FT) također ima visoke prosječne plate, sa prosjekom od 153,214 USD.
   - **Početnici (EN)** i **srednji nivo (MI)** imaju značajno niže plate u svim tipovima zaposlenja u poređenju sa starijim nivoom i ekspertima.
   - Ugovorno zaposlenje (CT) i puno radno vrijeme (FT) imaju veće plate u poređenju sa slobodnim radnicima (FL) i zaposlenicima sa skraćenim radnim vremenom (PT).

Zaključak:
- Eksperti (EX) i stariji nivo (SE) imaju značajno više plate, posebno u ugovornom zaposlenju i punom radnom vremenu.
- Početnici i srednji nivo imaju niže plate, ali razlike među njima su manje izražene u poređenju sa starijim nivoom i ekspertima.
- Ugovorno zaposlenje i puno radno vrijeme nude konkurentnije plate u poređenju sa slobodnim radnicima i zaposlenicima sa skraćenim radnim vremenom.


**Pitanje br. 13.**

*Da li analiza trenda prosječnih plata tokom posmatranog perioda ukazuje na rastući, opadajući ili stagnirajući trend?*

In [ ]:
work_2020 = df.loc[(df['work_year'] == 2020)]
work_2021 = df.loc[(df['work_year'] == 2021)]
work_2022 = df.loc[(df['work_year'] == 2022)]
work_2023 = df.loc[(df['work_year'] == 2023)]

hist_data = [work_2020['salary_in_usd'], work_2021['salary_in_usd'],
            work_2022['salary_in_usd'], work_2023['salary_in_usd']]
group_labels = ['2020 salary', '2021 salary', '2022 salary', '2023 salary']

year_salary = pd.DataFrame(columns = ['2020', '2021', '2022', '2023'])
year_salary['2020'] = work_2020.groupby('work_year').mean('salary_in_usd')['salary_in_usd'].values
year_salary['2021'] = work_2021.groupby('work_year').mean('salary_in_usd')['salary_in_usd'].values
year_salary['2022'] = work_2022.groupby('work_year').mean('salary_in_usd')['salary_in_usd'].values
year_salary['2023'] = work_2023.groupby('work_year').mean('salary_in_usd')['salary_in_usd'].values

fig1 = ff.create_distplot(hist_data, group_labels, show_hist = False)
fig2 = go.Figure(data=px.bar(x = year_salary.columns,
                            y = year_salary.values.tolist()[0],
                            color = year_salary.columns,
                            title = 'Mean Salary by Work Year'))

fig1.update_layout(title = 'Salary Distribution based on Work Year')
fig2.update_layout(xaxis_title = "Work Year", yaxis_title = "Mean Salary (k)")
fig1.show()
fig2.show()

Analiza:
1. **Distribucija plata po godinama rada:**
   - Gornji graf prikazuje distribuciju plata za različite godine rada (2020, 2021, 2022, 2023).
   - Donji graf prikazuje prosječnu platu za svaku godinu rada.

2. **Detalji:**
   - **2023**: Najviša prosječna plata od 149,046 USD, što pokazuje rast plata u odnosu na prethodne godine.
   - **2022**: Druga najviša prosječna plata od 133,339 USD, također pokazuje rast u odnosu na 2021. i 2020. godinu.
   - **2021** i **2020**: Niže prosječne plate u poređenju sa 2022. i 2023. godinom, ali su relativno blizu jedna drugoj.
   - Generalno, postoji trend rasta plata tokom godina od 2020. do 2023.

Zaključak:
- Prosječne plate su rasle svake godine od 2020. do 2023. godine.
- Najveći skok u prosječnim platama se vidi između 2021. i 2022. godine, a zatim se rast nastavlja do 2023. godine.
- Ovaj trend može ukazivati na poboljšanje ekonomskih uslova, povećanu potražnju za radnicima, ili promjene u tržištu rada.

*Pandemija COVID-19 imala je značajan uticaj na tržište rada, a sektor nauke o podacima nije bio izuzetak. Tokom pandemije, mnoge kompanije su ubrzale svoje digitalne transformacije, što je dovelo do veće potražnje za Data Scientistima i stručnjacima za analitiku. Kao rezultat toga, kompanije su bile spremne ponuditi veće plate kako bi privukle i zadržale ove ključne stručnjake. Pored toga, prelazak na rad na daljinu omogućio je većem broju ljudi da pristupe globalnim tržištima rada, što je dodatno povećalo konkurenciju za talente i dovelo do rasta plata.*


#CHI-SQUARE TEST
*Koristeći **CHI-SQUARE TEST** za ispitivanje nezavisnosti kategoričkih varijabli ispitati zavisnost/nezavisnost između nekoliko kategoričkih varijabli*

**1.0** *Da li postoji zavisnost između tipa zaposlenja i pola zaposlenih?*



In [ ]:
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df['employment_type'], df['Sex'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print("Chi-square Test:", chi2)
print("Vrijednost p:", p)


Vrijednost p je veća od uobičajene razine značajnosti od 0.05. To ukazuje na to da nema dovoljno statističkih dokaza da odbacimo nultu hipotezu. Na temelju ovih rezultata, ne možemo zaključiti da postoji statistički značajna veza između tipa zaposlenja i pola zaposlenih.



**1.1** *Da li postoji zavisnost između tipa zaposlenja i lokacije kompanije?*

In [ ]:
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df['employment_type'], df['company_location'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print("Chi-square Test:", chi2)
print("Vrijednost p:", p)


Vrijednost p je izuzetno mala (približno nula), što ukazuje na to da postoji vrlo mala vjerovatnoća da je ova zapažena zavisnost rezultat slučajnosti. Stoga odbacujemo nultu hipotezu i prihvaćamo alternativnu hipotezu da postoji statistički značajna zavisnost između tipa zaposlenja i lokacije kompanije.
Na temelju ovih rezultata, zaključujemo da postoji statistički značajna veza između tipa zaposlenja i lokacije kompanije.

**1.2** *Da li postoji zavisnost između nivoa iskustva i tipa zaposlenja?*

In [ ]:
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df['experience_level'], df['employment_type'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print("Chi-square Test:", chi2)
print("Vrijednost p:", p)


Vrijednost p je izuzetno mala (približno nula), što ukazuje na to da postoji vrlo mala vjerovatnoća da je ova zapažena zavisnost između nivoa iskustva i tipa zaposlenja rezultat slučajnosti. Stoga odbacujemo nultu hipotezu i prihvaćamo alternativnu hipotezu da postoji statistički značajna zavisnost između nivoa iskustva i tipa zaposlenja. Ovi rezultati ukazuju na to da vrsta zaposlenja ovisi o razini iskustva, što implicira da postoji povezanost između ta dva faktora.


#KOLMOGOROV-SMIMOV I SHAPIROV-WILK TESTOVI
*Za svaku numeričku varijablu u setu podataka provjeriti da li prati normalnu distribuciju i to koristeći **KOLMOGOROV-SMIMOV I SHAPIROV-WILK** statističke testove i **QQ-PLOT** kao grafički test*

**KOLMOGOROV-SMIRNOV TEST**

 **1.0** Da li raspodjela plata u oblasti Data Science-a za 2023. godinu prati normalnu raspodjelu?

* Nulta hipoteza (H0): Raspodjela plata u oblasti Data Science-a za 2023. godinu prati normalnu raspodjelu.

* Alternativna hipoteza (H1): Raspodjela plata u oblasti Data Science-a za 2023. godinu ne prati normalnu raspodjelu.

In [ ]:
from scipy.stats import kstest, norm
salaries =pd.to_numeric(df['salary_in_usd'])
ks_statistic, p_value = kstest(salaries, norm.cdf)
print("Kolmogorov-Smirnov test za normalnost raspodjele plata:")
print("KS statistika =", ks_statistic)
print("p-vrednost =", p_value)


Kolmogorov-Smirnov test se koristi za poređenje dva uzorka kako bi se odredilo da li potiču iz iste distribucije.
Ovi rezultati ukazuju na to da postoji značajna razlika između empirijske distribucije plata i teoretske normalne distribucije. P-vrednost je vrlo niska (0.0), što znači da imamo dovoljno jakih dokaza da odbacimo nultu hipotezu o normalnoj distribuciji. To sugeriše da distribucija plata u oblasti Data Science-a za 2023. godinu nije normalna.


**1.1** Da li raspodjela godina radnog iskustva prati normalnu raspodjelu?

* Nulta hipoteza (H0): Raspodjela godina radnog iskustva je normalna.
* Alternativna hipoteza (H1): Raspodjela godina radnog iskustva nije normalna.

In [ ]:
from scipy.stats import kstest, norm
work_years = df['work_year']
ks_statistic, p_value = kstest(work_years, norm.cdf)
print("Kolmogorov-Smirnov test za normalnost raspodjele godina radnog iskustva:")
print("KS statistika =", ks_statistic)
print("p-vrijednost =", p_value)


**1.2** Da li raspodjela radnog vrijemena na daljinu prati normalnu raspodjelu?

  * Nulta hipoteza (H0): Raspodjela udjela radnog vrijemena na daljinu je normalna.
  * Alternativna hipoteza (H1): Raspodjela udjela radnog vrijemena na daljinu
  nije normalna.

In [ ]:
from scipy.stats import uniform
remote_ratio = df['remote_ratio']
ks_statistic_remote, p_value_remote = kstest(remote_ratio, norm.cdf)
print("Kolmogorov-Smirnov test za uniformnost raspodjele udjela rada na daljinu:")
print("KS statistika =", ks_statistic_remote)
print("p-vrijednost =", p_value_remote)


Budući da je p-vrijednost manja od uobičajene razine značajnosti (npr. 0.05), imamo dovoljno dokaza da odbacimo nultu hipotezu. To znači da raspodjela udjela radnog vremena na daljinu nije normalna. Na temelju ovih rezultata, zaključujemo da raspodjela udjela radnog vremena na daljinu ne slijedi normalnu raspodjelu.

**SHAPIRO-WILK TEST**


**Kao što je rečeno u postavci dva testa kojim se možemo služiti rade za numeričke varijable prema tome možemo pokazati kako uraditi proračun i preko Shapiro-Wilk testa**

**1.0** Da li distribucija plata u oblasti Data Science-a za 2023. godinu prati normalnu distribuciju?

   * Nulta hipoteza (H0): Distribucija plata u oblasti Data Science-a za 2023. godinu prati normalnu distribuciju.

   * Alternativna hipoteza (H1): Distribucija plata u oblasti Data Science-a za 2023. godinu ne prati normalnu distribuciju.



In [ ]:
from scipy.stats import kstest, norm
salaries =pd.to_numeric(df['salary_in_usd'])
statistic, p_value = shapiro(salaries)
print("Shapiro-Wilk Test:", statistic)
print("Shapiro-Wilk p-vrijednost:", p_value)

Možemo zaključiti i potvrditi već ranije izračunato- Ne prati normalnu distribuciju

**1.1** Da li distribucija godina radnog iskustva prati normalnu distribuciju?.

  * Nulta hipoteza (H0): Distribucija godina radnog iskustva je normalna.
  * Alternativna hipoteza (H1): Distribucija godina radnog iskustva nije normalna.

In [ ]:
from scipy.stats import kstest, norm
work_years = df['work_year']
statistic, p_value = shapiro(work_years)
print("Shapiro-Wilk Test:", statistic)
print("Shapiro-Wilk p-vrijednost:", p_value)




1.1 Da li distribucija godina radnog iskustva prati normalnu distribuciju?

  * Nulta hipoteza (H0): Distribucija godina radnog iskustva je normalna.
  * Alternativna hipoteza (H1): Distribucija godina radnog iskustva nije normalna.



In [ ]:
from scipy.stats import uniform
remote_ratio = df['remote_ratio']
statistic_remote, p_value_remote = shapiro(remote_ratio)
print("Shapiro-Wilk Test:", statistic)
print("Shapiro-Wilk p-vrijednost:", p_value)

#QQ-PLOT ->GRAFIČKI TEST

**1.0**  QQ (Quantile-Quantile) plot za prikazivanje distribucije plata zaposlenika:

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
salaries = df['salary_in_usd']
norm_dist = np.random.normal(loc=salaries.mean(), scale=salaries.std(), size=len(salaries))
norm_dist.sort()
salaries_sorted = np.sort(salaries)
plt.figure(figsize=(8, 6))
stats.probplot(salaries_sorted, dist="norm", plot=plt)
plt.title('QQ Plot za distribuciju plata')
plt.xlabel('Teorijski kvantili normalne distribucije')
plt.ylabel('Kvantili stvarnih plata')
plt.grid(True)
plt.show()


*1.Distribucija plata:*

*QQ plot prikazuje poređenje kvantila stvarnih plata zaposlenika (plave tačke) *sa teoretskim kvantilima normalne distribucije (crvena linija).*


*2.Odstupanje od normalne distribucije:*

*Većina tačaka prati crvenu liniju, što ukazuje da plate zaposlenika približno prate normalnu distribuciju.*
*Međutim, primjećujemo odstupanja na krajevima plota.*

*3.Značajna odstupanja:*

*Odstupanja na krajevima plota ukazuju na prisustvo ekstremnih vrednosti ili outlier-a, što znači da su neke plate znatno niže ili više od onoga što bi se očekivalo u normalnoj distribuciji.*
*To može biti rezultat veoma visokih plata za specifične pozicije ili veoma niskih plata za druge pozicije.*

*4.Zaključak:*

Plate zaposlenika generalno prate normalnu distribuciju, ali postoje značajna odstupanja na krajevima distribucije. Ova odstupanja sugerišu prisustvo outlier-a, koji mogu biti važni za detaljniju analizu i razumevanje distribucije plata u organizaciji.

# **Testiranje pouzdanosti za numeričke i kategorijske varijable**

**Testiranje intervala pouzdanosti za kategorijsku varijablu tip zaposljenja(employment type)**

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

confidence_level = 0.98
employment_type_counts = df['employment_type'].value_counts()
sample_size = len(df)
proportions = employment_type_counts / sample_size
std_devs = np.sqrt((proportions * (1 - proportions)) / sample_size)

z_value = stats.norm.ppf((1 + confidence_level) / 2)
lower_bounds = proportions - z_value * std_devs
upper_bounds = proportions + z_value * std_devs

results_df = pd.DataFrame({
    'Employment Type': employment_type_counts.index,
    'Proporcija': proportions,
    'Donja granica': lower_bounds,
    'Gornja granica': upper_bounds
}).reset_index(drop=True)

print("Intervali pouzdanosti proporcije za Employment Type:")
print(results_df.to_string(index=False))

*Na osnovu intervala pouzdanosti od 95% jasno možemo zaključiti da većinski dio zaposlenika je zaposleno kao "Full-time" tj rade puno radno vrijeme koje je predviđeno, dok ostali tipovi zaposljenja su jako malo zastupljeni.*

**Testiranje intervala pouzdanosti za kategorijsku varijablu tip posla(job title)**

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

confidence_level = 0.98
job_counts = df['job_title'].value_counts()
sample_size = len(df)
proportions = job_counts / sample_size
std_devs = np.sqrt((proportions * (1 - proportions)) / sample_size)

z_value = stats.norm.ppf((1 + confidence_level) / 2)
lower_bounds = proportions - z_value * std_devs
upper_bounds = proportions + z_value * std_devs

results_df = pd.DataFrame({
    'Job Title': job_counts.index,
    'Proporcija': proportions,
    'Donja granica': lower_bounds,
    'Gornja granica': upper_bounds
}).reset_index(drop=True)

print("Intervali pouzdanosti proporcije za tip posla:")
print(results_df.to_string(index=False))


*Na osnovu intervala pouzdanosti od 95%, možemo nedvosmisleno zaključiti da većinski dio radnih pozicija u ovoj kompaniji zauzimaju zaposlenici koji obavljaju uloge kao što su "Data Engineer", "Data Scientist" i "Data Analyst". Ovi rezultati ukazuju na to da kompanija preferira zaposlenike sa ovim specifičnim vještinama, dok su ostale radne pozicije manje zastupljene u odnosu na ove koje čine većinu radne snage kompanije.*

**Testiranje intervala pouzdanosti za kategorijsku varijablu lokacija kompanije
(company location)**

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

confidence_level = 0.95
company_location_counts = df['company_location'].value_counts()
sample_size = len(df)
proportions = company_location_counts / sample_size
std_devs = np.sqrt(proportions * (1 - proportions) / sample_size)

z_value = stats.norm.ppf((1 + confidence_level) / 2)
lower_bounds = proportions - z_value * std_devs
upper_bounds = proportions + z_value * std_devs

results_df = pd.DataFrame({
    'Company Location': company_location_counts.index,
    'Proporcija': proportions,
    'Donja granica': lower_bounds,
    'Gornja granica': upper_bounds
}).reset_index(drop=True)

print("Intervali pouzdanosti proporcije za Company Location:")
print(results_df.to_string(index=False))

*Na osnovu intervala pouzdanosti od 95% jasno možemo zaključiti da većinski dio kompanija je smješteno unutar zemlje "US" što sugerira da kompanije unutar "US" čine značajno većinski udio u promatranom setu podataka, dok ostale lokacije kao što su "GB", "CA", "ES" čine mali udio u setu podataka. Ostale lokacije ukazuju na jako malu prisustnost kompanije na toj lokaciji.*

**Testiranje intervala pouzdanosti za kategorijsku varijablu nivoa iskustva(experience level)**

In [ ]:
import pandas as pd
import scipy.stats as stats

confidence_level = 0.98
experience_level_counts = df['experience_level'].value_counts()
sample_size = len(df)
proportions = experience_level_counts / sample_size
std_devs = (proportions * (1 - proportions) / sample_size) ** 0.5

z_value = stats.norm.ppf((1 + confidence_level) / 2)
lower_bounds = proportions - z_value * std_devs
upper_bounds = proportions + z_value * std_devs

results_df = pd.DataFrame({
    'Experience Level': experience_level_counts.index,
    'Proporcija': proportions,
    'Donja granica': lower_bounds,
    'Gornja granica': upper_bounds
}).reset_index(drop=True)

print("Intervali pouzdanosti proporcije za Experience Level:")
print(results_df.to_string(index=False))

*Na osnovu intervala pouzdanosti, sa 95% sigurnošću možemo zaključiti da: Zaposlenici sa nivoom iskustva "Senior Executive" predstavljaju relativnu većinu u datom setu podataka. Zaposlenici sa nivoom iskustva "Mid-level/Intermediate" takođe čine značajan, ali manji deo ukupnog broja zaposlenih.Zaposlenici sa nivoom iskustva "Entry-Level" zauzimaju manji deo u odnosu na zaposlene sa nivoom iskustva "Senior Executive" i "Mid-level/Intermediate".
Zaposlenici sa najmanjom zastupljenošću u ovom delu analize su oni sa nivoom iskustva "Executive level", što ukazuje da izvršni direktori čine mali deo promatranog seta podataka.*

**Testiranje intervala pouzdanosti za kategorijsku varijablu veličina kompanije(company size)**

In [ ]:
import pandas as pd
import scipy.stats as stats


confidence_level = 0.98
company_size_counts = df['company_size'].value_counts()
sample_size = len(df)
proportions = company_size_counts / sample_size
std_devs = (proportions * (1 - proportions)) / sample_size
std_devs = std_devs ** 0.5

z_value = stats.norm.ppf((1 + confidence_level) / 2)
lower_bounds = proportions - z_value * std_devs
upper_bounds = proportions + z_value * std_devs

results_df = pd.DataFrame({
    "Velicina kompanije": company_size_counts.index,
    "Proporcija": proportions,
    "Donja granica": lower_bounds,
    "Gornja granica": upper_bounds
}).reset_index(drop=True)

print(results_df.to_string(index=False))

*Na osnovu intervala pouzdanosti (95% pouzdanost) jasno možemo zaključiti da veličina kompanije "Medium" (srednja veličina) je češća i čini veći udio u trenutnom setu podataka, veličina kompanija "Large" je manje česta i čini manji udio u trenutnom setu podataka i veličina kompanija "Small" čine jako mali dio trenutnog seta podataka.*

**Testiranje intervala pouzdanosti za numeričke varijable**

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import io
import requests

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
numeric_df = df[numeric_columns]

confidence_level = 0.95
mean = numeric_df.mean()
std = numeric_df.std(ddof=1)
sample_size = len(numeric_df)
t_value = stats.t.ppf((1 + confidence_level) / 2, df=sample_size - 1)
standard_error = std / np.sqrt(sample_size)

lower_bound = mean - t_value * standard_error
upper_bound = mean + t_value * standard_error

print("Interval pouzdanosti srednje vrijednosti:")
print("Donja granica:\n", lower_bound)
print("\n")
print("Gornja granica:\n", upper_bound)

  * Godina rada (work_year):
      Zaključak: Srednja vrijednost godine rada se nalazi unutar ovog intervala sa određenim stepenom pouzdanosti (najčešće 95%). To znači da je prosečna godina rada za uzorak vrlo blizu 2022. godine, sa malim varijacijama.

  * Plata (salary):
      Interval pouzdanosti: Od 169205.230037 do 212185.913505
      Zaključak: Srednja vrijednost plate se nalazi unutar ovog intervala. Ovo pokazuje da prosečna plata u uzorku varira između otprilike 169,205 i 212,186, što može ukazivati na značajne razlike u platama unutar uzorka.

  * Plata u USD (salary_in_usd):
      Interval pouzdanosti: Od 135552.920258 do 139587.859502
      Zaključak: Srednja vrijednost plate u američkim dolarima je u rasponu od oko 135,553 do 139,588 USD. Ovaj interval je relativno uži u poređenju sa lokalnom platom, što može ukazivati na manju varijaciju plata kada se izraze u USD.

  * Udio rada na daljinu (remote_ratio):
      Interval pouzdanosti: Od 44.717027 do 47.826248
      Zaključak: Srednja vrijednost udela rada na daljinu je između 44.7% i 47.8%. To sugeriše da je približno polovina posla obavljena na daljinu u prosjeku za uzorak.



# Hipoteze:

## **Hipoteza 1:**
* Ne postoji statistički značajna razlika u prosječnoj plati između zaposlenih sa različitim nivoima iskustva.

In [ ]:
from scipy.stats import f_oneway

entry_salaries = df[df['experience_level'] == 'EN']['salary_in_usd']
mid_salaries = df[df['experience_level'] == 'MI']['salary_in_usd']
senior_salaries = df[df['experience_level'] == 'SE']['salary_in_usd']
exec_salaries = df[df['experience_level'] == 'EX']['salary_in_usd']

f_statistic, p_value = f_oneway(entry_salaries, mid_salaries, senior_salaries, exec_salaries)

print("P-vrijednost:", p_value)

if p_value < 0.05:
    print("Hipoteza se odbija. Postoji statistički značajna razlika u prosječnoj plaći između različitih nivoa iskustva.")
else:
    print("Hipoteza se prihvaća. Nema statistički značajne razlike u prosječnoj plaći između različitih nivoa iskustva.")



Zaključak:Na temelju provedene analize podataka, može se zaključiti da postoji statistički značajna razlika u prosječnoj plaći između zaposlenih s različitim nivoima iskustva. To znači da zaposlenici s višim nivoima iskustva prosječno zarađuju više od zaposlenika s nižim nivoima iskustva.

## **Hipoteza 2:**
* Udaljenosti radnika se razlikuje u različitim vrstama zaposlenja.

In [ ]:
from itertools import combinations
from scipy.stats import chi2_contingency

employment_groups = df.groupby('employment_type')
results = []

for group1, group2 in combinations(employment_groups, 2):
    count1 = group1[1][group1[1]['remote_ratio'] == 100].shape[0]
    count2 = group2[1][group2[1]['remote_ratio'] == 100].shape[0]
    nobs1 = group1[1]['remote_ratio'].count()
    nobs2 = group2[1]['remote_ratio'].count()

    contingency_table = [[count1, nobs1 - count1], [count2, nobs2 - count2]]
    contingency_table_corrected = [[cell + 0.5 for cell in row] for row in contingency_table]

    chi2, p_value, _, _ = chi2_contingency(contingency_table_corrected)

    result = {'Group 1': group1[0], 'Group 2': group2[0], 'p-value': p_value}
    results.append(result)

for result in results:
    group1 = result['Group 1']
    group2 = result['Group 2']
    p_value = result['p-value']
    print(f'Usporedba između {group1} i {group2}: p-value = {p_value}')
    if p_value < 0.05:
        print("Hipoteza se odbija. Postoji statistički značajna razlika u postotku udaljenih radnika između dvije vrste zaposlenja.")
    else:
        print("Hipoteza se prihvaća. Nema statistički značajne razlike u postotku udaljenih radnika između dvije vrste zaposlenja.")


Zaključak:

  * CT vs FL: Nema statistički značajne razlike u postotku udaljenih radnika između kontraktora (CT) i slobodnih profesionalaca (FL) (p-vrijednost = 0.1697).

  * CT vs FT: Postoji statistički značajna razlika u postotku udaljenih radnika između kontraktora (CT) i zaposlenika s punim radnim vremenom (FT) (p-vrijednost = 0.0107).

  * CT vs PT: Nema statistički značajne razlike u postotku udaljenih radnika između kontraktora (CT) i zaposlenika s nepunim radnim vremenom (PT) (p-vrijednost = 0.1488).

  * FL vs FT: Nema statistički značajne razlike u postotku udaljenih radnika između slobodnih profesionalaca (FL) i zaposlenika s punim radnim vremenom (FT) (p-vrijednost = 0.9007).

  * FL vs PT: Nema statistički značajne razlike u postotku udaljenih radnika između slobodnih profesionalaca (FL) i zaposlenika s nepunim radnim vremenom (PT) (p-vrijednost = 1.0).

  * FT vs PT: Nema statistički značajne razlike u postotku udaljenih radnika između zaposlenika s punim radnim vremenom (FT) i zaposlenika s nepunim radnim vremenom (PT) (p-vrijednost = 0.5842).

*Iz ovih rezultata možemo zaključiti da jedino postoji značajna razlika u postotku udaljenih radnika između kontraktora (CT) i zaposlenika s punim radnim vremenom (FT), dok ostale usporedbe nisu pokazale statistički značajne razlike. To sugerira da specifični radni statusi utječu na udaljenost radnika, ali većinom nema značajnih razlika.*


## **Hipoteza 3:**
H0: Ne postoji statistički značajna razlika u prosječnoj plati između dvije izabrane razine iskustva.

H1: Postoji statistički značajna razlika u prosječnoj plati između dvije izabrane razine iskustva.

In [ ]:
from itertools import combinations
import pandas as pd
from scipy import stats
experience_groups = df.groupby('experience_level')
results = []

for group1, group2 in combinations(experience_groups, 2):
    _, p_value = stats.ttest_ind(group1[1]['salary'], group2[1]['salary'])
    result = {'Group 1': group1[0], 'Group 2': group2[0], 'p-value': p_value}
    results.append(result)

for result in results:
    group1 = result['Group 1']
    group2 = result['Group 2']
    p_value = result['p-value']
    if p_value < 0.05:
        significance = "statistički značajna. Nulta hipoteza se odbacuje."
    else:
        significance = "nije statistički značajna. Nulta hipoteza se prihvaća."

    print(f"Razlika između {group1} i {group2} je {significance}")
    print(f"p-vrijednost: {p_value}")
    print()


Zaključak:

  * EN vs EX: Nema statistički značajne razlike u prosječnoj plati između početnika (EN) i iskusnih (EX) radnika. Nulta hipoteza se prihvaća (p-vrijednost = 0.3165).

  * EN vs MI: Nema statistički značajne razlike u prosječnoj plati između početnika (EN) i radnika sa srednjim iskustvom (MI). Nulta hipoteza se prihvaća (p-vrijednost = 0.4264).

  * EN vs SE: Nema statistički značajne razlike u prosječnoj plati između početnika (EN) i stručnjaka (SE). Nulta hipoteza se prihvaća (p-vrijednost = 0.3353).

  * EX vs MI: Nema statistički značajne razlike u prosječnoj plati između iskusnih (EX) radnika i radnika sa srednjim iskustvom (MI). Nulta hipoteza se prihvaća (p-vrijednost = 0.9910).

  * EX vs SE: Postoji statistički značajna razlika u prosječnoj plati između iskusnih (EX) radnika i stručnjaka (SE). Nulta hipoteza se odbacuje (p-vrijednost = 0.0075).

  * MI vs SE: Postoji statistički značajna razlika u prosječnoj plati između radnika sa srednjim iskustvom (MI) i stručnjaka (SE). Nulta hipoteza se odbacuje (p-vrijednost = 0.0050).

*Iz ovih rezultata možemo zaključiti da, osim u slučajevima usporedbe između iskusnih radnika (EX) i stručnjaka (SE) te radnika sa srednjim iskustvom (MI) i stručnjaka (SE), nema statistički značajnih razlika u prosječnoj plati među različitim razinama iskustva. To sugerira da, općenito, razina iskustva ne utječe značajno na prosječnu platu, osim kada se radi o stručnjacima u usporedbi s iskusnim radnicima ili radnicima sa srednjim iskustvom.*

*Ovi zaključci su temeljeni na p-vrijednostima dobivenim iz t-testa za neovisne uzorke. Za svaku kombinaciju razina iskustva , analizirali smo da li postoji statistički značajna razlika u prosječnim platama. Ako je p-vrijednost manja od 0.05, smatramo da je razlika statistički značajna i odbacujemo nultu hipotezu. U suprotnom, prihvaćamo nultu hipotezu i zaključujemo da nema značajne razlike u prosječnim platama između tih grupa.*

## **Hipoteza 4:**

H0: Ne postoji utjecaj faktora spola na iznos plaće zaposlenika.

H1: Postoji barem jedna razlika među srednjim vrijednostima plaća zaposlenika za različite vrijednosti faktora spola.

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('salary_in_usd ~ C(Sex)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)


Zaključak:
Na osnovu rezultata ANOVA tablice:

C(Sex): sum_sq = 1.617295e+10, df = 1.0, F = 4.070958, PR(>F) = 0.043698
P-vrijednost (PR(>F)) za faktor Sex je manja od 0.05, što sugerira da odbacujemo nul hipotezu (H0). To znači da postoji statistički značajna razlika u srednjim vrijednostima plata zaposlenika između različitih spolova. Drugim riječima, spol zaposlenika ima značajan utjecaj na iznos plate.



## **Hipoteza 5:**

 * Postoji značajna razlika u postotku udaljenih radnika između malih i velikih kompanija.













In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
from itertools import combinations

size_groups = df.groupby('company_size')
results = []

for group1, group2 in combinations(size_groups, 2):
    count1 = group1[1][group1[1]['remote_ratio'] == 1]['remote_ratio'].count()
    count2 = group2[1][group2[1]['remote_ratio'] == 1]['remote_ratio'].count()
    nobs1 = group1[1]['remote_ratio'].count()
    nobs2 = group2[1]['remote_ratio'].count()

    contingency_table = [[count1, nobs1-count1], [count2, nobs2-count2]]
    contingency_table_corrected = [[cell + 0.5 for cell in row] for row in contingency_table]

    chi2, p_value, _, _ = chi2_contingency(contingency_table_corrected)

    result = {'Group 1': group1[0], 'Group 2': group2[0], 'p-value': p_value}
    results.append(result)

for result in results:
    group1 = result['Group 1']
    group2 = result['Group 2']
    p_value = result['p-value']

    alpha = 0.05

    if p_value < alpha:
        print(f"Razlika između {group1} i {group2} je statistički značajna. Hipoteza se odbacuje.")
    else:
        print(f"Razlika između {group1} i {group2} nije statistički značajna. Hipoteza se prihvaća.")

Razlika u postotku udaljenih radnika između kompanija male (L) i srednje (M) veličine nije statistički značajna. Na temelju dostupnih podataka, nedostaje dovoljno dokaza koji bi podržali ideju o postojanju razlike između ovih skupina. Stoga se prihvata hipoteza o nedostatku razlike između kompanija male i srednje veličine.

Slično tome, razlika između kompanija male (L) i velike (S) veličine, kao i između kompanija srednje (M) i velike (S) veličine, takođe nije statistički značajna. Nakon analize dostupnih podataka, nedostaju dovoljno snažni dokazi koji bi podržali ideju o postojanju razlika između ovih skupina. Stoga se prihvata hipoteza o nedostatku razlike između kompanija male i velike veličine, kao i između kompanija srednje i velike veličine.

## **Hipoteza 6:**

H0: Ne postoji statistički značajna razlika u prosječnoj plaći između dviju izabranih lokacija.

H1: Postoji statistički značajna razlika u prosječnoj plaći između dviju izabranih lokacija.

In [ ]:
import pandas as pd
from scipy import stats
from itertools import combinations


location_groups = df.groupby('company_location')
results = []

for group1, group2 in combinations(location_groups, 2):
    _, p_value = stats.ttest_ind(group1[1]['salary'], group2[1]['salary'])
    result = {'Group 1': group1[0], 'Group 2': group2[0], 'p-value': p_value}
    results.append(result)

for result in results:
    group1 = result['Group 1']
    group2 = result['Group 2']
    p_value = result['p-value']

    if p_value < 0.05:
        significance = "statistički značajna. Nulta hipoteza se odbacuje."
    else:
        significance = "nije statistički značajna. Nulta hipoteza se prihvata."

    print(f"Razlika između {group1} i {group2} je {significance}.")
    print(f"Grupna p-vrijednost: {p_value}")
    print()


Kada p-vrijednost bude manja od 0.05, to znači da postoji statistički značajna razlika u prosječnim plaćama između dve ispitivane lokacije. U tim slučajevima, nulta hipoteza se odbacuje, što sugeriše da su prosječne plate na tim lokacijama zaista različite.

Kada p-vrijednost bude veća od 0.05, to znači da nema dovoljno dokaza za postojanje statistički značajne razlike u prosječnim plaćama između dve ispitivane lokacije. U tim slučajevima, nulta hipoteza se prihvata, što sugeriše da prosječne plate na tim lokacijama nisu značajno različite.

Iz rezultata možemo zaključiti gde postoji značajna razlika u prosečnim platama. U ovom primeru su analizirane sve moguće kombinacije lokacija. Ako određene lokacije nisu prisutne u rezultatima, to može značiti da imaju veoma niske p-vrednosti, zbog čega nije bilo moguće izračunati tačnu vrednost, ili da eventualno nedostaju neki podaci.

# **LINEARNA REGRESIJA** #

**1.Analiza plata na osnovu radnog iskustva i lokacije**

In [ ]:
import pandas as pd
import statsmodels.api as sm

X_categorical = df[['employment_type', 'job_title', 'company_location']]
X_encoded = pd.get_dummies(X_categorical, drop_first=True)
X = sm.add_constant(X_encoded)
print(X.dtypes)
y = df['salary_in_usd']
print(X.isnull().sum())
print(y.isnull().sum())
X = X.astype(float)
y = y.astype(float)
model = sm.OLS(y, X).fit()
print(model.summary())
predicted_values = model.predict(X)
mse = ((predicted_values - y) ** 2).mean()
r_squared = model.rsquared
print(f'MSE: {mse}')
print(f'R-squared: {r_squared}')

*R-kvadrat (R-squared) vrijednost iznosi 0.399, što znači da objašnjava samo 39.9% varijabilnosti u podacima. Ovo sugerira da model nije najbolje prilagođen podacima. Prilagođeni R-kvadrat (Adj. R-squared) vrijednost iznosi 0.390, što je slično R-kvadratu, ali uzima u obzir broj prediktora u modelu. Ovo sugerira da dodavanje prediktora nije značajno poboljšalo objašnjenje varijabilnosti. F-statistika iznosi 42.062, s p-vrijednosti manjom od 0.001. Ovo ukazuje na to da je ukupni model statistički značajan, tj. postoji značajna veza između prediktora i ciljne varijable.*

*Koeficijenti (coef) prikazuju utjecaj svakog prediktora na ciljnu varijablu (salary_in_usd) uzimajući u obzir sve ostale prediktore u modelu. Neki koeficijenti su statistički značajni (p-vrijednost manja od 0.05), dok su drugi neznačajni. Konstanta (const) ima vrijednost od 71970, što znači da je očekivana vrijednost ciljne varijable kada su svi prediktori nula 71970 USD.*

*Kao konačni zaključak, iako je ukupni model statistički značajan, njegova sposobnost objašnjenja varijabilnosti u podacima je ograničena. Dodatno poboljšanje modela možda bi bilo moguće kroz dodavanje dodatnih relevantnih prediktora ili poboljšanje kvalitete podataka.*

**2.Analiza utjecaja nivoa iskustva, tipa zaposlenja i spola na visinu plate**

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
label_encoder = LabelEncoder()

df["experience_level"] = label_encoder.fit_transform(df["experience_level"])
df["employment_type"] = label_encoder.fit_transform(df["employment_type"])
df["Sex"] = label_encoder.fit_transform(df["Sex"])
features = ["experience_level", "employment_type", "Sex"]
target = "salary_in_usd"
model = LinearRegression()
model.fit(X=df[features], y=df[target])
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
X = sm.add_constant(df[features])
ols_model = sm.OLS(df[target], X)
ols_results = ols_model.fit()
print(ols_results.summary())



*R-kvadrat (R-squared) vrijednost od 0.108 sugerira da se oko 10.8% varijabilnosti ciljne varijable (salary_in_usd) može objasniti pomoću prediktora u modelu. Međutim, to znači da postoji značajan dio varijabilnosti koji nije objašnjen modelom, što ukazuje na to da model nije jako dobar u objašnjavanju varijabilnosti plaća.*

*Adjustirani R-kvadrat (Adj. R-squared) vrijednost od 0.107 uzima u obzir broj prediktora i broj promatranja. Iako je ova vrijednost vrlo slična R-kvadratu, ona je korisniji pokazatelj kvalitete modela s obzirom na broj varijabli.*

*F-statistika ima vrijednost od 151.7, a p-vrijednost je statistički značajna (8.55e-93), što ukazuje na to da je barem jedan od prediktora značajan za objašnjavanje varijabilnosti u ciljnoj varijabli. Ovo znači da model kao cjelina značajno objašnjava varijabilnost plaća, unatoč niskom R-kvadratu.*

*Koeficijenti prediktora pokazuju da:*

*Experience_level ima pozitivan i značajan koeficijent (22771.4, p < 0.001), što znači da viši nivo iskustva značajno povećava plaću.
Employment_type ima negativan i nesignifikantan koeficijent (-171.8, p = 0.931), što znači da tip zaposlenja nema značajan utjecaj na plaću.
Sex ima negativan i značajan koeficijent (-4297.1, p < 0.001), što znači da postoji statistički značajan negativan efekt spola na plaću.
Durbin-Watson statistika od 1.754 sugerira da nema značajne autokorelacije u rezidualima.*

*Jarque-Bera test za normalnost reziduala je statistički značajan (p < 0.001), što sugerira da postoji značajno odstupanje od normalnosti. Ovo može utjecati na validnost regresijskih rezultata.*

*AIC i BIC vrijednosti su relativno visoke (93326.04 i 93326.04, respektivno), što može ukazivati na to da model nije najbolje prilagođen podacima.*

*Na temelju ovih rezultata, možemo zaključiti da, iako model pokazuje statističku značajnost, objašnjava vrlo mali dio varijabilnosti u plaćama i možda nije najbolji model za predviđanje plaća na temelju odabranih prediktora. Prediktori iskustvo i spol su značajni, dok tip zaposlenja nije značajan faktor.*

**3.Analiza stvarnog i predvidenog omjera rada na daljinu**

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
X = df_encoded.drop('remote_ratio', axis=1)
y = df_encoded['remote_ratio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [ ]:
plt.scatter(y_test, y_pred)
plt.xlabel('Stvarne vrijednosti')
plt.ylabel('Predviđene vrijednosti')
plt.title('Scatter plot: Stvarne vrijednosti vs. Predviđene vrijednosti (remote_ratio)')
plt.show()

In [ ]:
residuals = y_test - y_pred
plt.hist(residuals, bins=20)
plt.xlabel('Reziduali')
plt.ylabel('Broj uzoraka')
plt.title('Histogram reziduala')
plt.show()

In [ ]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error (MSE):", mse)
print("R-squared (R2):", r2)

Nakon provedenih analiza, razvili smo model linearne regresije za predikciju omjera rada na daljinu (remote_ratio) koristeći dostupne značajke iz dataset-a. Model smo trenirali na 80% podataka, dok smo 20% podataka iskoristili za testiranje.

Vizualizacija rezultata
Scatter plot koji prikazuje stvarne vrijednosti naspram predviđenih vrijednosti pokazuje kako model uspijeva u nekim slučajevima precizno predvidjeti remote_ratio, dok su u drugim slučajevima predikcije manje precizne. Histogram reziduala pokazuje da većina reziduala ima vrijednosti blizu nule, što upućuje na to da je model u prosjeku dobar, ali postoje određene predikcije s većim odstupanjem.

Evaluacijske metrike
Izračunate metričke vrijednosti za model su:
Mean Squared Error (MSE): 2204.7073967834144
R-squared (R2): 0.05550052107385528

Na osnovu ovih vrijednosti možemo zaključiti da model linearne regresije nije dobro prilagođen podacima, s obzirom na nisku vrijednost R² koja sugeriše da model ne objašnjava većinu varijabilnosti. Međutim, vrijednost MSE je relativno niska, što znači da postoji određena razlika između stvarnih i predviđenih vrijednosti, ali ta razlika nije velika.

# KONAČNI ZAKLJUČAK

*Kao tim, proveli smo detaljnu analizu skupa podataka o platama data naučnika za 2023. godinu dostupnog na Kaggleu (Data Science Salaries 2023). Cilj ove analize bio je da se dobije uvid u faktore koji utječu na plate u ovoj oblasti, te identificirati ključne determinante visokih zarada. U ovoj studiji, analizirali smo različite aspekte zapošljavanja, uključujući geografske lokacije, nivo obrazovanja, godine iskustva, specifične vještine i sektore industrije.*

##Izbor skupa podataka:
Ovaj skup podataka je odabran iz nekoliko ključnih razloga:

###Ažurnost:
Podaci su iz 2023. godine, što znači da pružaju relevantne i ažurirane informacije o trenutnom tržištu rada za data naučnike. Ova ažurnost omogućava precizne analize trenutnih trendova i obrazaca u zapošljavanju i plaćama.

###Obuhvatnost:
Skup podataka pokriva širok spektar zemalja, lokacija, pozicija i iskustva. Ovo omogućava analizu različitih faktora koji utječu na plate, pružajući širi kontekst i bolji uvid u globalne i regionalne varijacije.
###Kvaliteta podataka:
Jedan od ključnih faktora prilikom izbora skupa podataka bila je kvaliteta podataka. Analizirali smo strukturu, čistoću i kompletnost podataka kako bismo osigurali da su informacije koje koristimo relevantne i pouzdane. Skup podataka s Kagglea obično prolazi kroz proces procjene i validacije od strane zajednice, što doprinosi visokoj kvaliteti podataka.
###Dostupnost i pristupačnost:
Dostupnost podataka je također važan faktor, posebno ako želimo omogućiti reproduktivnost naše analize ili omogućiti drugima da koriste iste podatke za daljnja istraživanja.
###Širok spektar varijabli:
Skup podataka koji smo odabrali sadrži raznolike varijable koje su relevantne za analizu platnih razlika među data naučnicima. Osim osnovnih informacija o platama, uključene su i informacije o obrazovanju, iskustvu, lokaciji i specifičnim vještinama. Obrazovni nivo igra značajnu ulogu u određivanju visine plate, pri čemu osobe sa višim stepenom obrazovanja često imaju tendenciju da ostvaruju više plate.

##Najvažniji faktori za visoku platu:
Naša analiza je pokazala da su sljedeći faktori najvažniji za visoku platu data naučnika:

### Iskustvo:
Iskustvo je jedan od najvažnijih faktora koji utječu na visinu plate. Data naučnici s više godina iskustva generalno zarađuju više. Na primjer, senior data naučnici i oni na rukovodećim pozicijama obično imaju značajno veće plate u odnosu na početnike i juniore.

### Obrazovanje:
Viši nivoi obrazovanja, poput magistarske i doktorske titule u područjima kao što su računarstvo, statistika ili srodne discipline, često rezultiraju višim platama. Naša analiza je pokazala da posjedovanje naprednih akademskih kvalifikacija može biti značajan faktor u postizanju više zarade.
### Specifične vještine:
Vještine u području dubokog učenja, strojnog učenja, obrade prirodnog jezika (NLP) i rada s velikim podacima su visoko tražene i dobro nagrađene. Ove specifične vještine omogućavaju data naučnicima da preuzimaju složenije projekte i time opravdavaju više plate.
### Vrsta radnog mjesta:
Vrsta radnog mjesta također igra ključnu ulogu. Na primjer, direktori data nauke i savjetnici obično zarađuju više od data naučnika na početničkim i srednjim nivoima. Pozicije koje zahtijevaju strateško planiranje i upravljanje često imaju veće plate zbog povećane odgovornosti i složenosti posla.
### Industrija:
Industrija u kojoj data naučnik radi može značajno utjecati na visinu plate. Data naučnici u tehnološkim i financijskim sektorima obično imaju više plate nego oni u drugim industrijama. Ovi sektori često imaju veće budžete za analizu podataka i cijene stručnost koja može donijeti konkurentsku prednost.
##Utisci i budući pravci:

Na temelju naše analize, možemo zaključiti da je tržište rada za data naučnike dinamično i da postoji niz faktora koji utječu na plate. Iskustvo, obrazovanje i specifične vještine su ključni za postizanje visokih zarada. Osim toga, potražnja za data naučnicima raste u svim sektorima, što znači da će se ova profesija vjerovatno i dalje dobro isplatiti u budućnosti.

Istraživanje razlika u platama između različitih kompanija i geografskih područja:
Dublja analiza razlika u platama između različitih kompanija, kao i geografske varijacije, mogla bi otkriti dodatne uvide. Na primjer, razlike između urbanih i ruralnih područja ili između različitih zemalja i kontinenata.

Detaljna analiza utjecaja specifičnih vještina i certifikata na plate:
Istraživanje kako specifične vještine i profesionalni certifikati utječu na visinu plate moglo bi pružiti korisne informacije za one koji razmišljaju o razvoju karijere u ovom polju. Na primjer, koje certifikate i kurseve poslodavci najviše cijene.

Usporedba plata data naučnika s platama u drugim profesijama u IT sektoru:
Poređenje plata data naučnika sa drugim profesijama u IT sektoru može pomoći u razumijevanju relativne vrijednosti ove profesije. Ovo može uključivati usporedbe s programerima, inženjerima za internet sigurnost, analitičarima podataka i drugim srodnim profesijama.

*Vjerujemo da će nam ove dodatne analize dati još dublje razumijevanje tržišta rada za data naučnike i pomoći pojedincima da donose informirane odluke o svojim karijerama. Naša analiza naglašava važnost kontinuiranog usavršavanja i obrazovanja, kao i prilagođavanja promjenama u tehnološkim trendovima kako bi se ostalo konkurentno na tržištu rada.*




Zadaću radili:

1.Samira Ćatić 100%

2.Lina Mašović 100%

3.Nedim Omanović 100%

4.Bilal Ozdić 100%